# Avro Basics — 02: Writing Avro Files


Writing Avro correctly ensures downstream readers can deserialize your data.
Key decisions: schema definition, compression, and record naming.

Topics: format("avro"), compression codecs, avroSchema on write, recordName/namespace.


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 13:05:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Avro JAR: spark-avro_2.13-4.0.2


## Step 1 — Basic Write



In [2]:

import random, datetime, pathlib
random.seed(42)
EVENTS = ["page_view","click","purchase","search"]
PAGES  = ["/home","/products","/cart"]

df_src = spark.range(50_000).select(
    F.col("id").alias("event_id"),
    (F.col("id") % 1000 + 1).alias("user_id"),
    F.element_at(F.array([F.lit(e) for e in EVENTS]), (F.col("id")%4+1).cast("int")).alias("event_type"),
    F.element_at(F.array([F.lit(p) for p in PAGES]),  (F.col("id")%3+1).cast("int")).alias("page"),
    F.when(F.col("id")%7==0, F.round(F.rand(42)*500,2)).otherwise(F.lit(0.0)).alias("revenue"),
    F.current_timestamp().alias("event_ts"),
)

OUT = f"{DATA_DIR}/write_test"
df_src.write.format("avro").mode("overwrite").save(OUT)
import glob as glib
files = glib.glob(f"{OUT}/*.avro")
size_kb = sum(pathlib.Path(f).stat().st_size for f in files)//1024
print(f"Written: {df_src.count():,} rows | {len(files)} file(s) | {size_kb} KB")


Written: 50,000 rows | 4 file(s) | 597 KB


## Step 2 — Compression Codecs

Avro supports: uncompressed, snappy, deflate, bzip2, xz.

In [3]:

codecs = ["uncompressed", "snappy", "deflate", "bzip2"]
results = {}

for codec in codecs:
    path = f"{DATA_DIR}/codec_{codec}"
    t0 = time.time()
    df_src.write.format("avro").mode("overwrite") \
          .option("compression", codec).save(path)
    t_write = time.time() - t0
    files = glib.glob(f"{path}/*.avro")
    size_kb = sum(pathlib.Path(f).stat().st_size for f in files)//1024
    results[codec] = {"write": t_write, "size_kb": size_kb}

base = results["uncompressed"]["size_kb"]
print(f"{'Codec':<15} {'Write':>8} {'Size KB':>10} {'vs uncompressed':>16}")
print("-" * 54)
for codec, r in results.items():
    ratio = f"{r['size_kb']/base:.2f}x"
    print(f"  {codec:<13} {r['write']:>6.2f}s {r['size_kb']:>8} KB {ratio:>16}")


Codec              Write    Size KB  vs uncompressed
------------------------------------------------------
  uncompressed    0.72s     1960 KB            1.00x
  snappy          0.66s      597 KB            0.30x
  deflate         0.63s      305 KB            0.16x
  bzip2           0.89s      189 KB            0.10x


## Step 3 — Writing with Explicit Avro Schema

Control record name, namespace, and field docs.

In [4]:

WRITE_SCHEMA = """{
  "type": "record",
  "name": "UserEvent",
  "namespace": "com.example.events",
  "doc": "User interaction event",
  "fields": [
    {"name": "event_id",   "type": "long",   "doc": "Unique event ID"},
    {"name": "user_id",    "type": "long",   "doc": "User identifier"},
    {"name": "event_type", "type": "string", "doc": "Type of event"},
    {"name": "page",       "type": "string", "doc": "Page URL"},
    {"name": "revenue",    "type": "double", "doc": "Revenue, 0 if non-purchase"},
    {"name": "event_ts",   "type": {"type":"long","logicalType":"timestamp-micros"}, "doc": "Event timestamp"}
  ]
}"""

SCHEMA_PATH = f"{DATA_DIR}/with_schema"
df_src.write.format("avro") \
            .mode("overwrite") \
            .option("avroSchema", WRITE_SCHEMA) \
            .save(SCHEMA_PATH)

files = glib.glob(f"{SCHEMA_PATH}/*.avro")
print("Written with explicit schema:")
df_back = spark.read.format("avro").load(SCHEMA_PATH)
df_back.printSchema()
df_back.show(3)


Written with explicit schema:
root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- event_ts: timestamp (nullable = true)

+--------+-------+----------+---------+-------+--------------------+
|event_id|user_id|event_type|     page|revenue|            event_ts|
+--------+-------+----------+---------+-------+--------------------+
|   12500|    501| page_view|    /cart|    0.0|2026-04-09 13:05:...|
|   12501|    502|     click|    /home|    0.0|2026-04-09 13:05:...|
|   12502|    503|  purchase|/products| 400.88|2026-04-09 13:05:...|
+--------+-------+----------+---------+-------+--------------------+
only showing top 3 rows


## Step 4 — Write Modes and coalesce



In [5]:

TABLE = f"{DATA_DIR}/rw_modes"

df_src.coalesce(1).write.format("avro").mode("overwrite").save(TABLE)
print(f"overwrite: {spark.read.format('avro').load(TABLE).count():,} rows, 1 file")

df_src.limit(1000).write.format("avro").mode("append").save(TABLE)
print(f"append   : {spark.read.format('avro').load(TABLE).count():,} rows (1000 added)")

df_src.limit(500).write.format("avro").mode("ignore").save(TABLE)
print(f"ignore   : {spark.read.format('avro').load(TABLE).count():,} rows (unchanged)")


overwrite: 50,000 rows, 1 file
append   : 51,000 rows (1000 added)
ignore   : 51,000 rows (unchanged)


## Summary



In [6]:

print("""
df.write.format("avro")
        .mode("overwrite")
        .option("compression", "snappy")   # uncompressed|snappy|deflate|bzip2|xz
        .option("avroSchema", json_str)    # explicit Avro schema
        .save("/path/output/")

Single file:
  df.coalesce(1).write.format("avro").mode("overwrite").save(path)

Compression guide:
  snappy    — balanced (recommended)
  deflate   — higher ratio, slower
  bzip2     — highest ratio, slowest
  uncompressed — fastest, no compression
""")



df.write.format("avro")
        .mode("overwrite")
        .option("compression", "snappy")   # uncompressed|snappy|deflate|bzip2|xz
        .option("avroSchema", json_str)    # explicit Avro schema
        .save("/path/output/")

Single file:
  df.coalesce(1).write.format("avro").mode("overwrite").save(path)

Compression guide:
  snappy    — balanced (recommended)
  deflate   — higher ratio, slower
  bzip2     — highest ratio, slowest
  uncompressed — fastest, no compression

